### Clone Repo

In [ ]:
USER = "github.com/AIML-ORG"
REPO = "modded-nanogpt"
BRANCH = "aman"

try:
    from kaggle_secrets import UserSecretsClient
    import os

    TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN_AIML")

    REPO_PATH = os.path.join("/kaggle/working", REPO)
    REPO_URL = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"

    if os.path.exists(REPO_PATH):
        os.chdir(REPO_PATH)

        print("Repo exists")

        current_branch = os.popen("git rev-parse --abbrev-ref HEAD").read().strip()

        if current_branch == BRANCH:
            print("Updating current branch...")
            !git pull --depth 1 origin {BRANCH}
        else:
            print(f"Switching branch to {BRANCH}...")
            # Make the new branch visible to the shallow clone
            !git remote set-branches origin {BRANCH}
            # Fetch only the difference
            !git fetch --depth 1 origin {BRANCH}
            # Switch and track
            !git checkout {BRANCH}
            # %cd {REPO}
    else:
        print("Cloning repo")
        !git clone --depth 1 -b {BRANCH} --single-branch -j4 https://{TOKEN}@{USER}/{REPO}.git

        os.chdir(REPO_PATH)

except Exception as e:
    print(e)
    print(f"Not in Kaggle environment. error: {e}")

### Install dependencies

In [ ]:
!pip install -U -q torch torchvision torchaudio
!pip install -q -r requirements.txt

### Load data

In [ ]:
!python data/cached_fineweb10B.py 9

### Start training

In [9]:
# !torchrun --standalone --nproc_per_node=8 train_gpt.py
!torchrun --standalone train_gpt.py

^C
W0114 19:33:59.574000 245 torch/distributed/elastic/agent/server/api.py:725] Received 2 death signal, shutting down workers
W0114 19:33:59.575000 245 torch/distributed/elastic/multiprocessing/api.py:908] Sending process 250 closing signal SIGINT
[rank0]: Traceback (most recent call last):
[rank0]:   File "/kaggle/working/modded-nanogpt/train_gpt.py", line 2023, in <module>
[rank0]:     model(inputs, targets, cum_seqlens, ws_long // 2, ws_long).backward()
[rank0]:     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py", line 414, in __call__
[rank0]:     return super().__call__(*args, **kwargs)
[rank0]:            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
[rank0]:     return self._call_impl(*args, **kwargs)
[rank0]:            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]:   File "/usr/lo

In [ ]:
!torchrun --standalone train_gpt.py

In [ ]:
import torch.distributed as dist
import torch

device = torch.device("cuda", int(os.environ["LOCAL_RANK"]))
torch.cuda.set_device(device)
dist.init_process_group(backend="nccl", device_id=device)
dist.barrier()
print(dist.get_world_size())